# Part1 Knowledge base construction

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
!mkdir -p /content/data
!cp /content/drive/MyDrive/PE6201_Course_Project/data/*.pdf /content/data/
!ls /content/data | wc -l

26


In [12]:
!pip install -q pymupdf tqdm sentence-transformers faiss-cpu numpy rank_bm25 openai tiktoken

## The function of this code block
1. Read 26 PDF files, extract text, and clean it (Extract & Clean).
2. Split the text by page to form individual chunks (Chunking).
3. Assign Paragraph IDs and metadata (the most critical step).
4. Save the result in JSONL format (generate chunks.jsonl).

In [13]:
import fitz
import re
import json
from pathlib import Path
from tqdm import tqdm

DATA_DIR = Path("/content/data")
OUTPUT_FILE = "/content/chunks.jsonl"

def parse_filename(fname: str):
    # Matches Class 2-6: PE6201_ClassX_CY_...
    m = re.search(r'PE6201_Class(\d+)_(C\d+|Close)_', fname)
    if m:
        return f"Class {m.group(1)}", m.group(2)
    # Matches Class 1: PE6201_CY_...
    m = re.search(r'PE6201_(C\d+)_', fname)
    if m:
        return "Class 1", m.group(1)
    return None, None

def clean_text(text):
    lines = text.splitlines()
    cleaned = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        # Filter out pure numeric lines (e.g., 1 1 1 1)
        if re.fullmatch(r"[\d\s]+", line):
            continue
        # Filter out long lines with high digit density
        if len(line) > 100 and len(re.findall(r"\d", line)) / len(line) > 0.6:
            continue
        cleaned.append(line)
    return "\n".join(cleaned)

def extract_chunks(pdf_path):
    fname = pdf_path.name
    class_name, capsule = parse_filename(fname)
    if not class_name or not capsule:
        print(f"[Skipping] Could not parse filename: {fname}")
        return []

    prefix = f"CL{class_name.split()[-1]}_{capsule}"
    doc = fitz.open(pdf_path)
    chunks = []

    for page_num, page in enumerate(doc, start=1):
        text = clean_text(page.get_text("text"))
        if len(text.strip()) < 20:
            continue
        chunks.append({
            "paragraph_id": None,
            "class": class_name,
            "capsule": capsule,
            "source": fname,
            "page": page_num,
            "text": text
        })

    for i, ch in enumerate(chunks, start=1):
        ch["paragraph_id"] = f"{prefix}_P{i:03d}"
    return chunks

all_chunks = []
pdf_files = sorted(DATA_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files")

for pdf in tqdm(pdf_files):
    all_chunks.extend(extract_chunks(pdf))

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for ch in all_chunks:
        f.write(json.dumps(ch, ensure_ascii=False) + "\n")

print(f"\nGenerated {len(all_chunks)} chunks")
print(f"Written to {OUTPUT_FILE}")

Found 26 PDF files


100%|██████████| 26/26 [00:02<00:00,  8.74it/s]


Generated 425 chunks
Written to /content/chunks.jsonl


In [14]:
# Copy to Google Drive
!cp /content/chunks.jsonl /content/drive/MyDrive/PE6201_Course_Project/

# Confirm whether the copy was successful
!ls -lh /content/drive/MyDrive/PE6201_Course_Project/chunks.jsonl

-rw------- 1 root root 411K Oct  4 00:43 /content/drive/MyDrive/PE6201_Course_Project/chunks.jsonl


## Quality Check
1. Verify data integrity (Is there any file loss?)
2. Check core functionality (Is paragraph ID generation working correctly?)
3. Assess the text cleaning results (Is there any noise?)

In [15]:
import json

chunks = [json.loads(l) for l in open("/content/chunks.jsonl", encoding="utf-8")]

print(f"Total chunks: {len(chunks)}")
print(f"First chunk: {chunks[0]['paragraph_id']}")
print(f"Last chunk: {chunks[-1]['paragraph_id']}")
print()
print("First 3 chunks preview:")
for ch in chunks[:3]:
    print(f"--- {ch['paragraph_id']} | {ch['class']} | {ch['capsule']} | page {ch['page']}")
    print(ch["text"][:200])
    print()

Total chunks: 425
First chunk: CL1_C1_P001
Last chunk: CL6_Close_P007

First 3 chunks preview:
--- CL1_C1_P001 | Class 1 | C1 | page 1
C1
Your intuition is wrong
26 minutes
BY THE END OF THIS CAPSULE YOU WILL BE ABLE TO
Explain why you cannot predict what AI is good at by asking how hard the task
looks.
MSc in Enterprise Artificial I

--- CL1_C1_P002 | Class 1 | C1 | page 2
Before I teach you anything,
let's find out what you already believe.
There are no marks for being right.
PE6201  ·  EMERGING AI TECHNOLOGIES
Class 1  ·  C1 — Your intuition is wrong
MSc in Enterprise

--- CL1_C1_P003 | Class 1 | C1 | page 3
POLL 1  ·  THE FRONTIER CHECK
Which of these can today's best AI do reliably?
Select all you think it can.
Win a gold medal at the Maths
Olympiad
Read an analog clock
Predict a protein's 3D structure




## Establish vector indices


1. Generate Embeddings
2. Create FAISS Index
3. Provide the Retrieval Layer of the RAG system



In [16]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import json
from tqdm import tqdm

CHUNKS_FILE = "/content/chunks.jsonl"
INDEX_FILE = "/content/index.faiss"
EMBED_MODEL = "all-MiniLM-L6-v2"

# Load chunks
chunks = [json.loads(l) for l in open(CHUNKS_FILE, encoding="utf-8")]
texts = [c["text"] for c in chunks]
print(f"Loaded {len(texts)} chunks")

# Generate embeddings
print("Generating embeddings... this may take 1-2 minutes.")
model = SentenceTransformer(EMBED_MODEL)
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True
).astype(np.float32)

# Build FAISS index
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# Save index
faiss.write_index(index, INDEX_FILE)
print(f"\nIndex saved: {INDEX_FILE}")
print(f"Total vectors: {index.ntotal}")

Loaded 425 chunks
Generating embeddings... this may take 1-2 minutes.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]


Index saved: /content/index.faiss
Total vectors: 425


In [17]:
!cp /content/index.faiss /content/drive/MyDrive/PE6201_Course_Project/
!ls -lh /content/drive/MyDrive/PE6201_Course_Project/index.faiss

-rw------- 1 root root 638K Oct  4 00:45 /content/drive/MyDrive/PE6201_Course_Project/index.faiss


## Test retrieval

In [18]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load index and chunks
model = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.read_index("/content/index.faiss")
chunks = [json.loads(l) for l in open("/content/chunks.jsonl", encoding="utf-8")]

# Test query
query = "What is RAG?"
q_emb = model.encode([query], normalize_embeddings=True).astype(np.float32)
scores, ids = index.search(q_emb, 5) # Get top 5

print(f"Query: {query}\n")
for score, idx in zip(scores[0], ids[0]):
    ch = chunks[idx]
    print(f"score={score:.3f} | {ch['paragraph_id']} | {ch['source']} p{ch['page']}")
    print(ch["text"][:250])
    print("---")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Query: What is RAG?

score=0.477 | CL2_C3_P014 | PE6201_Class2_C3_RAG.pdf p14
Q U I C K C H E C K 2 · W O O C L A P
In RAG, what does the embedding model produce?
A)  A vector — numbers that capture meaning
B)  A short summary of the text
C)  The final answer to the user
D)  A keyword index
Vote in Wooclap — answer on the next
---
score=0.455 | CL2_C3_P009 | PE6201_Class2_C3_RAG.pdf p9
PE6201  ·  EMERGING AI TECHNOLOGIES
Class 2 · C3 — In-context learning & RAG
When to RAG — and when not to
The routing decision: don't reach for RAG reflexively.
MSc in Enterprise Artificial Intelligence   ·   T1 AY2026–27
Situation
Reach for
Private
---
score=0.454 | CL2_C3_P013 | PE6201_Class2_C3_RAG.pdf p13
QUICK CHECK 1 · ANSWER
C — retrieval.
Retrieval fetches the relevant chunks and injects them into the prompt — the core of RAG (and what the
Colab builds).
---
score=0.399 | CL2_C3_P012 | PE6201_Class2_C3_RAG.pdf p12
Q U I C K C H E C K 1 · W O O C L A P
In a RAG system, what sits between the user'

# Part2 Evaluation & Benchmarking
## Evaluation

In [19]:
eval_set = [
    # Class 1
    {"qid": "Q001", "question": "What is the Jagged Frontier?", "gold_paragraph_id": "CL1_C1_P009", "class": "Class 1"},
    {"qid": "Q002", "question": "Why can't AI count the letters in the word strawberry?", "gold_paragraph_id": "CL1_C3_P014", "class": "Class 1"},
    {"qid": "Q003", "question": "What is the difference between machines that sort and machines that make?", "gold_paragraph_id": "CL1_C2_P003", "class": "Class 1"},
    {"qid": "Q004", "question": "What are the three stages of training a foundation model?", "gold_paragraph_id": "CL1_C3_P008", "class": "Class 1"},
    {"qid": "Q005", "question": "Why did Moderna build over 1,400 custom GPTs?", "gold_paragraph_id": "CL1_C4_P005", "class": "Class 1"},
    {"qid": "Q006", "question": "What are the four questions to ask before choosing an AI approach?", "gold_paragraph_id": "CL1_C5_P004", "class": "Class 1"},
    {"qid": "Q007", "question": "What is the two-file mental model of a model?", "gold_paragraph_id": "CL1_C3_P002", "class": "Class 1"},
    {"qid": "Q008", "question": "What did Moderna's DoseID GPT do?", "gold_paragraph_id": "CL1_C4_P007", "class": "Class 1"},

    # Class 2
    {"qid": "Q009", "question": "What are the seven layers of the modern AI stack?", "gold_paragraph_id": "CL2_C1_P007", "class": "Class 2"},
    {"qid": "Q010", "question": "What are the five build-vs-buy factors?", "gold_paragraph_id": "CL2_C2_P002", "class": "Class 2"},
    {"qid": "Q011", "question": "What is the difference between fine-tuning and RAG?", "gold_paragraph_id": "CL2_C3_P002", "class": "Class 2"},
    {"qid": "Q012", "question": "What are the two knobs you actually tune in RAG?", "gold_paragraph_id": "CL2_C3_P006", "class": "Class 2"},
    {"qid": "Q013", "question": "What are the three levels of evaluation?", "gold_paragraph_id": "CL2_C4_P004", "class": "Class 2"},
    {"qid": "Q014", "question": "What did DBS own versus rent in their AI stack?", "gold_paragraph_id": "CL2_C2_P008", "class": "Class 2"},
    {"qid": "Q015", "question": "What is the P-U-R-E governance framework?", "gold_paragraph_id": "CL2_C2_P008", "class": "Class 2"},
    {"qid": "Q016", "question": "What is the first upgrade when basic RAG plateaus?", "gold_paragraph_id": "CL2_C3_P007", "class": "Class 2"},

    # Class 3
    {"qid": "Q017", "question": "What is tokenization in the context of large language models?", "gold_paragraph_id": "CL3_C1_P003", "class": "Class 3"},
    {"qid": "Q018", "question": "What does attention do in a transformer model?", "gold_paragraph_id": "CL3_C1_P010", "class": "Class 3"},
    {"qid": "Q019", "question": "What is the prompting ladder?", "gold_paragraph_id": "CL3_C2_P003", "class": "Class 3"},
    {"qid": "Q020", "question": "Why is structured output important for AI systems?", "gold_paragraph_id": "CL3_C2_P009", "class": "Class 3"},
    {"qid": "Q021", "question": "What are the four meters running on every model call?", "gold_paragraph_id": "CL3_C3_P003", "class": "Class 3"},
    {"qid": "Q022", "question": "What is the one-line cost-to-serve formula?", "gold_paragraph_id": "CL3_C3_P005", "class": "Class 3"},
    {"qid": "Q023", "question": "Why does Med-PaLM 2's recipe cost 20-30x per answer?", "gold_paragraph_id": "CL3_C3_P014", "class": "Class 3"},
    {"qid": "Q024", "question": "How do you calculate the break-even success rate for AI architectures?", "gold_paragraph_id": "CL3_C3_P011", "class": "Class 3"},

    # Class 4
    {"qid": "Q025", "question": "What is an agent, in one line?", "gold_paragraph_id": "CL4_C1_P002", "class": "Class 4"},
    {"qid": "Q026", "question": "What two conditions must hold for a system to be an agent?", "gold_paragraph_id": "CL4_C1_P002", "class": "Class 4"},
    {"qid": "Q027", "question": "What is the difference between a workflow and an agent?", "gold_paragraph_id": "CL4_C1_P008", "class": "Class 4"},
    {"qid": "Q028", "question": "What are the three agent patterns?", "gold_paragraph_id": "CL4_C1_P010", "class": "Class 4"},
    {"qid": "Q029", "question": "Where should guardrails live, and what two kinds are there?", "gold_paragraph_id": "CL4_C2_P008", "class": "Class 4"},
    {"qid": "Q030", "question": "Why do eight turns cost more than eight calls in an agent?", "gold_paragraph_id": "CL4_C3_P002", "class": "Class 4"},
    {"qid": "Q031", "question": "What are the three tests to decide if an agent loop is worth the tokens?", "gold_paragraph_id": "CL4_C3_P005", "class": "Class 4"},
    {"qid": "Q032", "question": "What is the lethal trifecta in prompt injection?", "gold_paragraph_id": "CL6_C2_P016", "class": "Class 6"}, # 注意：这题实际在 Class 6

    # Class 5
    {"qid": "Q033", "question": "What are the two forces pulling AI costs in opposite directions?", "gold_paragraph_id": "CL5_C2_P003", "class": "Class 5"},
    {"qid": "Q034", "question": "What is cost per successful task?", "gold_paragraph_id": "CL5_C2_P004", "class": "Class 5"},
    {"qid": "Q035", "question": "What are the three layers of the cost-to-serve calculator?", "gold_paragraph_id": "CL5_C2_P021", "class": "Class 5"},
    {"qid": "Q036", "question": "What are the two archetypes of AI entering an industry?", "gold_paragraph_id": "CL5_C3_P002", "class": "Class 5"},
    {"qid": "Q037", "question": "What is the precondition for the first archetype of AI entering an industry?", "gold_paragraph_id": "CL5_C3_P004", "class": "Class 5"},
    {"qid": "Q038", "question": "What is the free-label test?", "gold_paragraph_id": "CL5_C1_P017", "class": "Class 5"},
    {"qid": "Q039", "question": "What are the five data readiness tests?", "gold_paragraph_id": "CL5_C1_P011", "class": "Class 5"},
    {"qid": "Q040", "question": "What is the difference between a business model and an operating model?", "gold_paragraph_id": "CL5_C1_P007", "class": "Class 5"},

    # Class 6
    {"qid": "Q041", "question": "What is confabulation, and why is it a better word than hallucination?", "gold_paragraph_id": "CL6_C1_P006", "class": "Class 6"},
    {"qid": "Q042", "question": "What are the two questions that sort every AI failure?", "gold_paragraph_id": "CL6_C1_P010", "class": "Class 6"},
    {"qid": "Q043", "question": "What is the dangerous quadrant in AI failure?", "gold_paragraph_id": "CL6_C1_P010", "class": "Class 6"},
    {"qid": "Q044", "question": "What three things does a review step need?", "gold_paragraph_id": "CL6_C1_P009", "class": "Class 6"},
    {"qid": "Q045", "question": "What are the four operational layers of responsible deployment?", "gold_paragraph_id": "CL6_C1_P017", "class": "Class 6"},
    {"qid": "Q046", "question": "What is the lethal trifecta?", "gold_paragraph_id": "CL6_C2_P016", "class": "Class 6"},
    {"qid": "Q047", "question": "What is denial of wallet?", "gold_paragraph_id": "CL6_C2_P017", "class": "Class 6"},
    {"qid": "Q048", "question": "What are the five regulatory regimes and their governable objects?", "gold_paragraph_id": "CL6_C2_P009", "class": "Class 6"},
    {"qid": "Q049", "question": "What does 'compliant with what, binding on whom, from when' mean?", "gold_paragraph_id": "CL6_C2_P009", "class": "Class 6"},
    {"qid": "Q050", "question": "What are the six habits that outlive every model version?", "gold_paragraph_id": "CL6_Close_P006", "class": "Class 6"}
]

with open("/content/eval_set.jsonl", "w", encoding="utf-8") as f:
    for item in eval_set:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

!cp /content/eval_set.jsonl /content/drive/MyDrive/PE6201_Course_Project/
print("✅ Successfully overwritten eval_set.jsonl with corrected gold_paragraph_id.")
print(f"Total questions: {len(eval_set)}")

✅ Successfully overwritten eval_set.jsonl with corrected gold_paragraph_id.
Total questions: 50


In [20]:
import json

# Load evaluation set and chunks
eval_set = [
    json.loads(l)
    for l in open("/content/eval_set.jsonl", encoding="utf-8")
]

chunks = [
    json.loads(l)
    for l in open("/content/chunks.jsonl", encoding="utf-8")
]

# Build lookup dictionary
chunk_dict = {
    ch["paragraph_id"]: ch
    for ch in chunks
}

print("===== Ground Truth Verification =====")

missing_ids = []

for item in eval_set:
    gold_id = item["gold_paragraph_id"]
    matched_chunk = chunk_dict.get(gold_id)

    if matched_chunk is None:
        missing_ids.append(gold_id)
        print(f"ERROR: {item['qid']} -> {gold_id} NOT FOUND")

print(f"\nTotal evaluation questions: {len(eval_set)}")
print(f"Missing Gold IDs: {len(missing_ids)}")

if len(missing_ids) == 0:
    print("✅ All 50 Gold Paragraph IDs exist in the knowledge base.")
else:
    print("❌ Some Gold Paragraph IDs are missing.")

# Preview a few verified examples
print("\n===== Sample Ground Truth =====")

for item in eval_set[:5]:
    ch = chunk_dict[item["gold_paragraph_id"]]

    print(f"\n{item['qid']}")
    print(f"Question: {item['question']}")
    print(f"Gold ID: {item['gold_paragraph_id']}")
    print(f"Course: {ch['class']}")
    print(f"Source: {ch['source']}")
    print(f"Page: {ch['page']}")
    print(f"Text: {ch['text'][:250]}...")

===== Ground Truth Verification =====

Total evaluation questions: 50
Missing Gold IDs: 0
✅ All 50 Gold Paragraph IDs exist in the knowledge base.

===== Sample Ground Truth =====

Q001
Question: What is the Jagged Frontier?
Gold ID: CL1_C1_P009
Course: Class 1
Source: PE6201_C1_Your_intuition_is_wrong.pdf
Page: 9
Text: THE JAGGED FRONTIER
AI capability does not track how hard a task looks to a human.
Ethan Mollick, 2024
Coined the term in Co-Intelligence — your first pre-
reading.
Stanford AI Index, 2026
Calls the same thing “jagged intelligence” —
Takeaway #4, two...

Q002
Question: Why can't AI count the letters in the word strawberry?
Gold ID: CL1_C3_P014
Course: Class 1
Source: PE6201_C3_What_scale_changed.pdf
Page: 14
Text: POLL  ·  CHECK THE MECHANISM
Why can't a model count the letters in “strawberry”?
A
It was never trained on spelling
B
It sees tokens, not letters — there is no character index
C
The word is too rare in training data
D
It can, but it chooses not to b...

Q003


## BM25 Baseline

In [21]:
import json
from rank_bm25 import BM25Okapi

# Load data
chunks = [json.loads(l) for l in open("/content/chunks.jsonl", encoding="utf-8")]
eval_set = [json.loads(l) for l in open("/content/eval_set.jsonl", encoding="utf-8")]

# Tokenize the corpus
tokenized_corpus = [ch["text"].lower().split() for ch in chunks]
bm25 = BM25Okapi(tokenized_corpus)

recall_1 = 0
recall_3 = 0

# Evaluate
for item in eval_set:
    tokenized_query = item["question"].lower().split()
    scores = bm25.get_scores(tokenized_query)

    # Get top 3 indices
    top_k_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:3]
    top_ids = [chunks[i]["paragraph_id"] for i in top_k_indices]

    if item["gold_paragraph_id"] in top_ids[:1]:
        recall_1 += 1
    if item["gold_paragraph_id"] in top_ids:
        recall_3 += 1

n = len(eval_set)
print(f"===== BM25 Baseline Evaluation =====")
print(f"Recall@1: {recall_1/n:.2%} ({recall_1}/{n})")
print(f"Recall@3: {recall_3/n:.2%} ({recall_3}/{n})")

===== BM25 Baseline Evaluation =====
Recall@1: 38.00% (19/50)
Recall@3: 62.00% (31/50)


## FAISS Semantic Retrieval

In [22]:
import json
import os
import numpy as np
import faiss
from openai import OpenAI
from google.colab import userdata
from tqdm import tqdm

# 1. 配置 API
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

# 2. 加载 chunks
chunks = [json.loads(l) for l in open("/content/chunks.jsonl", encoding="utf-8")]
texts = [c["text"] for c in chunks]

# 3. 调用 OpenAI 的 embedding 模型
print("Generating embeddings via API... This takes ~1 minute.")
embeddings = []
batch_size = 100
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    # 使用 text-embedding-3-small，注意处理超过模型最大长度的文本
    batch = [t[:8000] for t in batch]
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )
    embeddings.extend([data.embedding for data in response.data])

embeddings = np.array(embeddings).astype(np.float32)
# 归一化以便使用内积
faiss.normalize_L2(embeddings)

# 4. 重建 FAISS 索引
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, "/content/index_openai.faiss")

print(f"New index saved: /content/index_openai.faiss with {index.ntotal} vectors")
!cp /content/index_openai.faiss /content/drive/MyDrive/PE6201_Course_Project/

# 5. 用新的索引跑一次评估
eval_set = [json.loads(l) for l in open("/content/eval_set.jsonl", encoding="utf-8")]

def get_openai_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[text[:8000]]
    )
    emb = np.array(response.data[0].embedding).astype(np.float32)
    faiss.normalize_L2(emb.reshape(1, -1))
    return emb

recall_1, recall_3 = 0, 0
for item in eval_set:
    q_emb = get_openai_embedding(item["question"]).reshape(1, -1)
    scores, ids = index.search(q_emb, 3)
    top_ids = [chunks[i]["paragraph_id"] for i in ids[0]]
    if item["gold_paragraph_id"] in top_ids[:1]: recall_1 += 1
    if item["gold_paragraph_id"] in top_ids: recall_3 += 1

n = len(eval_set)
print(f"\n===== OpenAI Embedding FAISS Results =====")
print(f"Recall@1: {recall_1/n:.2%} ({recall_1}/{n})")
print(f"Recall@3: {recall_3/n:.2%} ({recall_3}/{n})")

Generating embeddings via API... This takes ~1 minute.


100%|██████████| 5/5 [00:07<00:00,  1.41s/it]

New index saved: /content/index_openai.faiss with 425 vectors



===== OpenAI Embedding FAISS Results =====
Recall@1: 52.00% (26/50)
Recall@3: 86.00% (43/50)


In [24]:
!cp /content/index_openai.faiss /content/drive/MyDrive/PE6201_Course_Project/
!ls -lh /content/drive/MyDrive/PE6201_Course_Project/index_openai.faiss

-rw------- 1 root root 2.5M Oct  4 00:57 /content/drive/MyDrive/PE6201_Course_Project/index_openai.faiss


# Part3 Generation and Evaluation

In [25]:
import os
import json
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# ================= 1. Configure API =================
from google.colab import userdata
try:
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
except userdata.SecretNotFoundError:
    raise ValueError("Please add your OPENROUTER_API_KEY to Colab Secrets (click the 🔑 icon on the left).")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# Choose a model
MODEL_NAME = "openai/gpt-4o-mini"

# ================= 2. Load chunks =================
print("Loading embedding model and FAISS index...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.read_index("/content/index.faiss")
chunks = [json.loads(l) for l in open("/content/chunks.jsonl", encoding="utf-8")]
eval_set = [json.loads(l) for l in open("/content/eval_set.jsonl", encoding="utf-8")]

# ================= 3. Generate OpenAI embeddings =================
def retrieve(query, top_k=3):
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q_emb, top_k)
    return [{"score": float(s), "paragraph_id": chunks[i]["paragraph_id"], "text": chunks[i]["text"]} for s, i in zip(scores[0], ids[0])]

def generate_answer(query, retrieved, threshold=0.3):
    if not retrieved or retrieved[0]["score"] < threshold:
        return "I couldn't find a reliable answer.", True # True means abstained

    context = "\n\n".join([f"[{r['paragraph_id']}] {r['text'][:800]}" for r in retrieved])
    prompt = f"""You are a course assistant. Answer the question based ONLY on the provided context.
If the context does not contain the answer, reply exactly: "I couldn't find a reliable answer."
You MUST cite the source paragraph ID for every answer. Format: Source: <paragraph_id>.

Context:
{context}

Question: {query}
Answer:"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content, False

def hallucination_check(answer, retrieved):
    # Abstention is not a hallucination
    if "I couldn't find a reliable answer." in answer:
        return False

    # Extract cited paragraph IDs
    cited_ids = re.findall(r'Source:\s*(CL\d+_\w+_P\d+)', answer)

    # No citation -> hallucination
    if not cited_ids:
        return True

    # Get the text of the cited sources
    cited_texts = [
        r["text"].lower()
        for r in retrieved
        if r["paragraph_id"] in cited_ids
    ]

    # Citation does not match retrieved context
    if not cited_texts:
        return True

    # Remove common stopwords
    stopwords = {
        "the", "a", "an", "is", "are", "was", "were",
        "of", "to", "in", "on", "for", "and", "or",
        "with", "that", "this", "it", "as", "by",
        "from", "be", "can", "may", "what", "how",
        "which", "does", "do"
    }

    answer_words = {
        w for w in re.findall(r'\b[a-zA-Z0-9]+\b', answer.lower())
        if w not in stopwords
    }

    # Check whether the cited source contains enough meaningful
    # words from the answer
    for txt in cited_texts:
        source_words = {
            w for w in re.findall(r'\b[a-zA-Z0-9]+\b', txt)
            if w not in stopwords
        }

        overlap = answer_words & source_words

        # Require both:
        # 1. At least 3 meaningful words overlap
        # 2. At least 20% of answer keywords are supported
        if len(overlap) >= 3 and len(overlap) / max(len(answer_words), 1) >= 0.20:
            return False   # Grounded

    return True   # Hallucination

# ================= 4. Rebuild FAISS index =================
abstained_count = 0
hallucination_count = 0
results = []

print("Running evaluation on 50 questions... this may take 1-2 minutes.")
for item in eval_set:
    query = item["question"]
    retrieved = retrieve(query, top_k=3)
    answer, abstained = generate_answer(query, retrieved, threshold=0.3)

    if abstained:
        abstained_count += 1
        is_hallucination = False # Abstaining is safe
    else:
        is_hallucination = hallucination_check(answer, retrieved)
        if is_hallucination:
            hallucination_count += 1

    item["generated_answer"] = answer
    item["abstained"] = abstained
    item["hallucination_flag"] = is_hallucination
    results.append(item)

# ================= 5. Save Results =================
with open("/content/rag_evaluation_results.jsonl", "w", encoding="utf-8") as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

!cp /content/rag_evaluation_results.jsonl /content/drive/MyDrive/PE6201_Course_Project/

# ================= 6. Print Summary =================
n = len(eval_set)
print("\n===== RAG Evaluation Summary =====")
print(f"Total questions evaluated: {n}")
print(f"Abstention rate: {abstained_count/n:.2%} ({abstained_count}/{n})")
print(f"Hallucination rate (among answered): {hallucination_count/(n-abstained_count):.2%} ({hallucination_count}/{n-abstained_count})")
print("\nExample result for Q001:")
print(f"Question: {results[0]['question']}")
print(f"Answer: {results[0]['generated_answer']}")
print(f"Abstained: {results[0]['abstained']} | Hallucination: {results[0]['hallucination_flag']}")

Loading embedding model and FAISS index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Running evaluation on 50 questions... this may take 1-2 minutes.

===== RAG Evaluation Summary =====
Total questions evaluated: 50
Abstention rate: 2.00% (1/50)
Hallucination rate (among answered): 53.06% (26/49)

Example result for Q001:
Question: What is the Jagged Frontier?
Answer: The Jagged Frontier refers to the concept that AI capability does not correlate with the perceived difficulty of a task. It highlights that the performance of AI can vary unpredictably based on the encoding of information and the emergence of capabilities at different scales. This means that to understand what an AI model can do, one must test it rather than rely on intuition about task difficulty. Source: [CL1_C1_P009], [CL1_C3_P010], [CL1_C5_P011].
Abstained: False | Hallucination: True


In [39]:
import os
import json
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from google.colab import userdata

# ================= 1. Configure OpenRouter =================
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
MODEL_NAME = "openai/gpt-4o-mini"

# ================= 2. Load Knowledge Base =================
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.read_index("/content/index.faiss")
chunks = [json.loads(l) for l in open("/content/chunks.jsonl", encoding="utf-8")]
eval_set = [json.loads(l) for l in open("/content/eval_set.jsonl", encoding="utf-8")]

# ================= 3. Define RAG Functions =================
def retrieve(query, top_k=3):
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q_emb, top_k)
    return [{"score": float(s), "paragraph_id": chunks[i]["paragraph_id"], "text": chunks[i]["text"]} for s, i in zip(scores[0], ids[0])]

def generate_answer(query, retrieved, threshold=0.3):
    if not retrieved or retrieved[0]["score"] < threshold:
        return "I couldn't find a reliable answer.", True # True means abstained

    context = "\n\n".join([f"[{r['paragraph_id']}] {r['text'][:800]}" for r in retrieved])

    # Core modification: Add explicit few-shot example to force the citation format
    prompt = f"""You are a course assistant. Answer the question based ONLY on the provided context.
If the context does not contain the answer, reply exactly: "I couldn't find a reliable answer."
You MUST cite the source paragraph ID for every answer.

Example of correct format:
Question: What is RAG?
Context: [CL2_C3_P003] RAG = embed -> retrieve -> generate.
Answer: RAG is a method that combines retrieval with generation. Source: CL2_C3_P003

Context:
{context}

Question: {query}
Answer:"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content, False

def hallucination_check(answer, retrieved):
    # Check if the answer contains a citation
    cited_ids = re.findall(r'Source:\s*(CL\d+_\w+_P\d+)', answer)
    if not cited_ids:
        return True # Flag as hallucination if no citation is found

    cited_texts = [r["text"].lower() for r in retrieved if r["paragraph_id"] in cited_ids]
    if not cited_texts:
        return True

    # Check word overlap between the answer and the cited chunk
    answer_words = set(re.findall(r'\w+', answer.lower()))
    for txt in cited_texts:
        txt_words = set(re.findall(r'\w+', txt))
        if len(answer_words & txt_words) >= 3:
            return False # Grounded

    return True # Hallucination if cited text does not overlap with answer

# ================= 4. Evaluate on 50 QA =================
abstained_count = 0
hallucination_count = 0
results = []

print("Running evaluation on 50 questions... this may take 1-2 minutes.")
for item in eval_set:
    query = item["question"]
    retrieved = retrieve(query, top_k=3)
    answer, abstained = generate_answer(query, retrieved, threshold=0.3)

    if abstained:
        abstained_count += 1
        is_hallucination = False
    else:
        is_hallucination = hallucination_check(answer, retrieved)
        if is_hallucination:
            hallucination_count += 1

    item["generated_answer"] = answer
    item["abstained"] = abstained
    item["hallucination_flag"] = is_hallucination
    results.append(item)

# ================= 5. Save Results =================
with open("/content/rag_evaluation_results.jsonl", "w", encoding="utf-8") as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

!cp /content/rag_evaluation_results.jsonl /content/drive/MyDrive/PE6201_Course_Project/

# ================= 6. Print Summary =================
n = len(eval_set)
print("\n===== RAG Evaluation Summary =====")
print(f"Total questions evaluated: {n}")
print(f"Abstention rate: {abstained_count/n:.2%} ({abstained_count}/{n})")
print(f"Hallucination rate (among answered): {hallucination_count/(n-abstained_count):.2%} ({hallucination_count}/{n-abstained_count})")
print("\nExample result for Q001:")
print(f"Question: {results[0]['question']}")
print(f"Answer: {results[0]['generated_answer']}")
print(f"Abstained: {results[0]['abstained']} | Hallucination: {results[0]['hallucination_flag']}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Running evaluation on 50 questions... this may take 1-2 minutes.

===== RAG Evaluation Summary =====
Total questions evaluated: 50
Abstention rate: 2.00% (1/50)
Hallucination rate (among answered): 46.94% (23/49)

Example result for Q001:
Question: What is the Jagged Frontier?
Answer: The Jagged Frontier refers to the concept that AI capability does not align with the perceived difficulty of tasks, indicating that one must test AI models to understand their capabilities. It highlights two mechanisms: representation, where the model's understanding is limited by its encoding, and emergence, where capabilities appear unpredictably with scale. Source: CL1_C1_P009, CL1_C3_P010.
Abstained: False | Hallucination: False


# Part4 cost analysis

In [40]:
import tiktoken
import json

# Assume gpt-4o-mini pricing
PRICE_IN = 0.15 / 1_000_000  # $0.15 per 1M input tokens
PRICE_OUT = 0.60 / 1_000_000 # $0.60 per 1M output tokens

enc = tiktoken.get_encoding("cl100k_base")

chunks = [json.loads(l) for l in open("/content/chunks.jsonl", encoding="utf-8")]
total_tokens = sum(len(enc.encode(c["text"])) for c in chunks)

# Estimate query cost: assume average 800 input tokens + 200 output tokens per query
avg_input_tokens = 800
avg_output_tokens = 200
cost_per_query = (avg_input_tokens * PRICE_IN) + (avg_output_tokens * PRICE_OUT)

print("===== Cost Analysis =====")
print(f"Total corpus tokens: {total_tokens}")
print(f"Corpus embedding cost (assuming $0.02/1M tokens): ${total_tokens / 1_000_000 * 0.02:.4f}")
print(f"Average tokens per query: {avg_input_tokens + avg_output_tokens}")
print(f"Average cost per query: ${cost_per_query:.6f}")
print(f"Cost per 1,000 queries: ${cost_per_query * 1000:.4f}")

===== Cost Analysis =====
Total corpus tokens: 89750
Corpus embedding cost (assuming $0.02/1M tokens): $0.0018
Average tokens per query: 1000
Average cost per query: $0.000240
Cost per 1,000 queries: $0.2400


In [41]:
!ls -lh /content/drive/MyDrive/PE6201_Course_Project/

total 1.1M
-rw------- 1 root root 411K Oct  3 18:40 chunks.jsonl
drwx------ 2 root root 4.0K Oct  3 14:37 data
-rw------- 1 root root 6.8K Oct  3 18:41 eval_set.jsonl
-rw------- 1 root root 638K Oct  3 18:41 index.faiss
-rw------- 1 root root  18K Oct  3 18:44 rag_evaluation_results.jsonl
